# Clickbait spoiler type classification: span tagging, then classification

SemEval 2023 Task 5, subtask 1 (predict `phrase`, `passage` or `multi`). This notebook tries a two-stage idea: first find where the spoiler is in the article, then use the found text to decide its type.

1. **Stage 1, span tagging.** `SpanBERT/spanbert-base-cased` is fine-tuned as a token classifier that marks the spoiler tokens in the article (B, I, O tags).
2. **Stage 2, classification.** The predicted spans are appended to the post, title and article text, and a `bert-base-uncased` classifier predicts the spoiler type.

The notebook defines the data handling, the datasets, the two models, the training loops and the pipeline, then runs it. The README of this folder gives the results and how they compare with the baselines.

## Setup and imports


In [ ]:
!pip install torch transformers pandas scikit-learn tqdm

In [ ]:
import json
import torch
import copy
import numpy as np
import pandas as pd
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report, balanced_accuracy_score
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from transformers import AutoConfig
from torch.optim import AdamW
from tqdm import tqdm

# Select the device used for computation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set paths and parameters
MAX_LEN = 256
TRAIN_BATCH_SIZE = 16
VALID_BATCH_SIZE = 32
EPOCHS = 3
LEARNING_RATE = 5e-6
RANDOM_SEED = 42
SPAN_MODEL_NAME = "SpanBERT/spanbert-base-cased"
BERT_MODEL_NAME = "bert-base-uncased"

# Set seeds for reproducibility
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

Using device: cuda


## Tokenizers

SpanBERT for stage 1 and `bert-base-uncased` for stage 2.


In [ ]:
# Load tokenizers
span_tokenizer = AutoTokenizer.from_pretrained(SPAN_MODEL_NAME)
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)

## Data preprocessing

The data gives each spoiler position as a paragraph index plus character offsets inside that paragraph. These functions convert them to absolute character positions in the paragraphs joined into one string, and then to token labels with the tokenizer's offset mapping: B (first token of a spoiler) is 1, I (inside) is 2 and O (outside) is 0.


In [ ]:
def load_data(file_path):
    """Load data from a JSONL file"""
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

def preprocess_spoiler_positions(spoiler_positions, target_paragraphs):
    """Convert spoiler positions from paragraph-relative positions
    to positions in the full text formed by joining all paragraphs"""

    # Validate the data first
    if not spoiler_positions or not target_paragraphs:
        return [], ""

    # Convert to a list if it is a string
    if isinstance(spoiler_positions, str):
        try:
            spoiler_positions = eval(spoiler_positions)
        except:
            return [], ""

    # Check that spoiler_positions has a valid format
    if not isinstance(spoiler_positions, list) or len(spoiler_positions) == 0:
        return [], ""

    full_text = " ".join(target_paragraphs)

    # Compute the length of each paragraph (including the space after it)
    para_lengths = [len(para) + 1 for para in target_paragraphs]  # +1 for the space
    para_start_idx = [0]

    # Compute the start position of each paragraph in the full text
    for i in range(len(para_lengths) - 1):
        para_start_idx.append(para_start_idx[-1] + para_lengths[i])

    absolute_positions = []

    # Check the format of spoiler_positions
    for pos in spoiler_positions:
        # Check that the format is valid
        if not isinstance(pos, list) or len(pos) != 2:
            continue

        # Check for the expected format [[para_idx, start_char], [para_idx, end_char]]
        if not all(isinstance(p, list) and len(p) == 2 for p in pos):
            continue

        # Extract the indices
        try:
            para_idx = pos[0][0]
            char_start = pos[0][1]
            char_end = pos[1][1]

            # Check that the indices are in range
            if para_idx >= len(target_paragraphs):
                continue

            # Compute absolute positions in the full text
            abs_start = para_start_idx[para_idx] + char_start
            abs_end = para_start_idx[para_idx] + char_end

            absolute_positions.append((abs_start, abs_end))
        except (IndexError, TypeError):
            continue

    return absolute_positions, full_text

def create_token_labels(full_text, absolute_positions, tokenizer):
    """Create token labels for span tagging using the BIO tagging scheme"""

    # Validate the data
    if not full_text or not absolute_positions:
        # Create an empty encoding of length MAX_LEN
        encoding = tokenizer(
            "",
            max_length=MAX_LEN,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        labels = torch.zeros(MAX_LEN, dtype=torch.long)
        encoding['labels'] = labels
        return encoding

    # Tokenize the text and get the offset mapping
    encoding = tokenizer(
        full_text,
        return_offsets_mapping=True,
        max_length=MAX_LEN,
        truncation=True,
        padding='max_length',
        return_tensors='pt'
    )

    offset_mapping = encoding.pop('offset_mapping')[0]

    # Create BIO labels (B: 1, I: 2, O: 0)
    labels = torch.zeros(len(offset_mapping), dtype=torch.long)

    for start_char, end_char in absolute_positions:
        # Find the tokens that match the spans
        token_start_idx = None
        token_end_idx = None

        for i, (token_start, token_end) in enumerate(offset_mapping):
            # Skip special tokens, which have offset (0, 0)
            if token_start.item() == 0 and token_end.item() == 0:
                continue

            # Find the first token of the span
            if token_start_idx is None and token_end.item() > start_char:
                token_start_idx = i

            # Find the last token of the span
            if token_start.item() < end_char:
                token_end_idx = i

            # Stop once we are past the end of the span
            if token_start.item() >= end_char:
                break

        # Skip spans that fall beyond the truncated length
        if token_start_idx is None or token_end_idx is None:
            continue

        # Label the first token B (Beginning)
        labels[token_start_idx] = 1

        # Label the remaining tokens in the span I (Inside)
        if token_end_idx > token_start_idx:
            labels[token_start_idx+1:token_end_idx+1] = 2

    encoding['labels'] = labels

    return encoding

## Datasets

`SpanTaggingDataset` holds the article tokens and their BIO labels for stage 1. `SpoilerClassificationDataset` builds the stage 2 input, `post - title - paragraphs [SPANS] predicted spans`, with the spoiler type as the label.


In [ ]:
class SpanTaggingDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=MAX_LEN):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.encodings = self.prepare_data()

    def prepare_data(self):
        encodings = []

        for item in tqdm(self.data, desc="Preparing span tagging data"):
            target_paragraphs = item.get('targetParagraphs', [])
            spoiler_positions = item.get('spoilerPositions', [])

            # Skip examples without spoiler positions
            if not spoiler_positions or not target_paragraphs:
                continue

            # Convert spoiler positions to absolute positions and join the paragraphs
            try:
                absolute_positions, full_text = preprocess_spoiler_positions(spoiler_positions, target_paragraphs)

                # Skip examples without absolute positions
                if not absolute_positions:
                    continue

                # Create token labels
                encoding = create_token_labels(full_text, absolute_positions, self.tokenizer)
                encodings.append(encoding)

            except Exception as e:
                print(f"Error processing item: {e}")
                continue

        return encodings

    def __len__(self):
        return len(self.encodings)

    def __getitem__(self, idx):
        # Make sure input_ids and attention_mask are tensors with the right dimensions
        input_ids = self.encodings[idx]['input_ids'].squeeze()
        attention_mask = self.encodings[idx]['attention_mask'].squeeze()
        labels = self.encodings[idx]['labels']

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels
        }


class SpoilerClassificationDataset(Dataset):
    def __init__(self, data, span_preds, tokenizer, max_len=MAX_LEN):
        self.data = data
        self.span_preds = span_preds  # Dictionary of span predictions keyed by uuid
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.encodings = self.prepare_data()

    def prepare_data(self):
        encodings = []
        label_map = {'phrase': 0, 'passage': 1, 'multi': 2}

        for item in tqdm(self.data, desc="Preparing classification data"):
            uuid = item.get('uuid')
            post_text = ' '.join(item.get('postText', []))
            target_title = item.get('targetTitle', '')
            target_paragraphs = item.get('targetParagraphs', [])
            label = item.get('tags')

            # Handle the case where the label is a list
            if isinstance(label, list) and len(label) > 0:
                label = label[0]  # Use the first value in the list

            # Join all the text
            full_text = f"{post_text} - {target_title} - {' '.join(target_paragraphs)}"

            # Add the span predictions (if any)
            span_info = ""
            if uuid in self.span_preds:
                spans = self.span_preds[uuid]
                span_info = " ".join(spans)
                full_text = f"{full_text} [SPANS] {span_info}"

            # Tokenize
            encoding = self.tokenizer(
                full_text,
                max_length=self.max_len,
                truncation=True,
                padding='max_length',
                return_tensors='pt'
            )

            # Add the label if it is a supported type
            if isinstance(label, str) and label in label_map:
                encoding['labels'] = torch.tensor(label_map[label])
                encodings.append(encoding)

        return encodings

    def __len__(self):
        return len(self.encodings)

    def __getitem__(self, idx):
        # Make sure input_ids and attention_mask are tensors with the right dimensions
        input_ids = self.encodings[idx]['input_ids'].squeeze()
        attention_mask = self.encodings[idx]['attention_mask'].squeeze()
        labels = self.encodings[idx]['labels']

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels
        }

## Models

`SpanTagger` is a SpanBERT token classifier. `SpoilerClassifier` is a BERT sequence classifier with dropout.


In [ ]:
class SpanTagger(nn.Module):
    def __init__(self, model_name, num_labels=3):  # 3 labels: O, B, I
        super(SpanTagger, self).__init__()
        self.model = AutoModelForTokenClassification.from_pretrained(model_name, num_labels=num_labels)

    def forward(self, input_ids, attention_mask, labels=None):
        return self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

    def predict(self, input_ids, attention_mask):
        with torch.no_grad():
            outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
            predictions = torch.argmax(outputs.logits, dim=2)
        return predictions


class SpoilerClassifier(nn.Module):
    def __init__(self, model_name, num_labels=3):
        super(SpoilerClassifier, self).__init__()
        config = AutoConfig.from_pretrained(model_name)
        config.num_labels = num_labels
        config.hidden_dropout_prob = 0.3  # Increase dropout
        config.attention_probs_dropout_prob = 0.3  # Increase dropout

        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            config=config
        )

    def forward(self, input_ids, attention_mask, labels=None):
        return self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

    def predict(self, input_ids, attention_mask):
        with torch.no_grad():
            outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
            predictions = torch.argmax(outputs.logits, dim=1)
        return predictions

## Training loops

The stage 2 loop uses early stopping on validation loss (patience 2) and restores the best weights.


In [ ]:
def train_span_tagger(model, train_dataloader, val_dataloader, epochs=EPOCHS, lr=LEARNING_RATE):
    optimizer = AdamW(model.parameters(), lr=lr)
    total_steps = len(train_dataloader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=total_steps
    )

    for epoch in range(epochs):
        model.train()
        train_loss = 0

        for batch in tqdm(train_dataloader, desc=f"Training Epoch {epoch+1}/{epochs}"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            model.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_dataloader)
        print(f"Epoch {epoch+1}/{epochs} - Average training loss: {avg_train_loss:.4f}")

        # Validation
        model.eval()
        val_loss = 0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch in tqdm(val_dataloader, desc=f"Validating Epoch {epoch+1}/{epochs}"):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss
                val_loss += loss.item()

                predictions = torch.argmax(outputs.logits, dim=2)

                # Collect predictions and labels for evaluation
                all_preds.extend(predictions.detach().cpu().numpy())
                all_labels.extend(labels.detach().cpu().numpy())

        avg_val_loss = val_loss / len(val_dataloader)
        print(f"Epoch {epoch+1}/{epochs} - Average validation loss: {avg_val_loss:.4f}")

        # Compute token-level accuracy, skipping padding tokens
        true_preds, true_labels = [], []
        for p, l, m in zip(all_preds, all_labels, batch['attention_mask'].cpu().numpy()):
            for i in range(len(m)):
                if m[i] == 1:  # Do not count padding
                    true_preds.append(p[i])
                    true_labels.append(l[i])

        accuracy = accuracy_score(true_labels, true_preds)
        print(f"Validation Token-level Accuracy: {accuracy:.4f}")

    return model


def train_spoiler_classifier(model, train_dataloader, val_dataloader, epochs=EPOCHS, lr=LEARNING_RATE):
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.01)  # Add weight decay
    total_steps = len(train_dataloader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=total_steps
    )

    best_val_loss = float('inf')
    best_model = None
    patience = 2  # Number of epochs to wait before stopping
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0

        for batch in tqdm(train_dataloader, desc=f"Training Epoch {epoch+1}/{epochs}"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            model.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_dataloader)
        print(f"Epoch {epoch+1}/{epochs} - Average training loss: {avg_train_loss:.4f}")

        # Validation
        model.eval()
        val_loss = 0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch in tqdm(val_dataloader, desc=f"Validating Epoch {epoch+1}/{epochs}"):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss
                val_loss += loss.item()

                predictions = torch.argmax(outputs.logits, dim=1)

                all_preds.extend(predictions.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        avg_val_loss = val_loss / len(val_dataloader)
        print(f"Epoch {epoch+1}/{epochs} - Average validation loss: {avg_val_loss:.4f}")

        # Compute accuracy and balanced accuracy
        accuracy = accuracy_score(all_labels, all_preds)
        balanced_acc = balanced_accuracy_score(all_labels, all_preds)

        print(f"Validation Accuracy: {accuracy:.4f}")
        print(f"Validation Balanced Accuracy: {balanced_acc:.4f}")
        print(classification_report(all_labels, all_preds, target_names=['phrase', 'passage', 'multi']))

        # Early stopping
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    # Load the best model if there is one
    if best_model is not None:
        model.load_state_dict(best_model)
        print(f"Loaded best model with validation loss: {best_val_loss:.4f}")

    return model

## From token predictions to spans

Turns runs of B and I predictions back into text spans, and generates the spans for every post. Stage 2 uses these spans as part of its input.


In [ ]:
def extract_spans_from_predictions(text, predictions, tokenizer, attention_mask):
    """Convert token predictions back into text spans"""

    offset_mapping = tokenizer(
        text,
        return_offsets_mapping=True,
        max_length=MAX_LEN,
        truncation=True,
        padding='max_length'
    )['offset_mapping']

    predictions = predictions.cpu().numpy()
    attention_mask = attention_mask.cpu().numpy()

    spans = []
    current_span = []
    in_span = False

    for i, (pred, mask, (start_offset, end_offset)) in enumerate(zip(predictions, attention_mask, offset_mapping)):
        # Skip padding and special tokens
        if mask == 0 or (start_offset == 0 and end_offset == 0):
            continue

        # B token (starts a new span)
        if pred == 1:
            if in_span:
                # If a span is already open, add it to the list of spans
                spans.append(current_span)
                current_span = []

            in_span = True
            current_span.append((start_offset, end_offset))

        # I token (continues the current span)
        elif pred == 2 and in_span:
            current_span.append((start_offset, end_offset))

        # O token (outside a span)
        elif pred == 0 and in_span:
            spans.append(current_span)
            current_span = []
            in_span = False

    # Add the last span (if any)
    if in_span and current_span:
        spans.append(current_span)

    # Convert token positions back into text
    text_spans = []
    for span in spans:
        if span:
            start_char = span[0][0]
            end_char = span[-1][1]

            if start_char < len(text) and end_char <= len(text):
                text_spans.append(text[start_char:end_char])

    return text_spans


def generate_span_predictions(model, data, tokenizer):
    """Generate span predictions from the SpanTagger model"""

    model.eval()
    span_predictions = {}

    for item in tqdm(data, desc="Generating span predictions"):
        uuid = item.get('uuid')
        target_paragraphs = item.get('targetParagraphs', [])

        if not target_paragraphs:
            continue

        full_text = " ".join(target_paragraphs)

        encoding = tokenizer(
            full_text,
            return_tensors='pt',
            max_length=MAX_LEN,
            truncation=True,
            padding='max_length'
        )

        input_ids = encoding['input_ids'].to(device)
        attention_mask = encoding['attention_mask'].to(device)

        with torch.no_grad():
            predictions = model.predict(input_ids, attention_mask)

        spans = extract_spans_from_predictions(
            full_text,
            predictions[0],
            tokenizer,
            attention_mask[0]
        )

        if spans:
            span_predictions[uuid] = spans

    return span_predictions


def evaluate_classifier(model, dataloader):
    """Evaluate the classifier model"""

    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            predictions = model.predict(input_ids, attention_mask)

            all_preds.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    balanced_acc = balanced_accuracy_score(all_labels, all_preds)

    print(f"Evaluation Accuracy: {accuracy:.4f}")
    print(f"Evaluation Balanced Accuracy: {balanced_acc:.4f}")
    print(classification_report(all_labels, all_preds, target_names=['phrase', 'passage', 'multi']))

    return balanced_acc


def predict_with_classifier(model, dataloader):
    """Predict with the classifier model"""

    model.eval()
    predictions = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Predicting"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            batch_preds = model.predict(input_ids, attention_mask)
            predictions.extend(batch_preds.cpu().numpy())

    return predictions

## Full pipeline

Trains stage 1, saves the tagger, generates spans for the train, validation and test posts, trains stage 2, and reports the validation scores. If a test file is given it also writes `submission.csv`.


In [ ]:
def run_full_pipeline(train_file, val_file, test_file=None):
    # Load data
    print("Loading data...")
    train_data = load_data(train_file)
    val_data = load_data(val_file)
    test_data = load_data(test_file) if test_file else None

    # Part 1: Fine-tune SpanBERT for span tagging
    print("\n=== Part 1: Fine-tuning SpanBERT for Span Tagging ===")

    # Create datasets
    train_span_dataset = SpanTaggingDataset(train_data, span_tokenizer)
    val_span_dataset = SpanTaggingDataset(val_data, span_tokenizer)

    print(f"Created span tagging datasets: {len(train_span_dataset)} training examples, {len(val_span_dataset)} validation examples")

    # Check that the datasets are not empty
    if len(train_span_dataset) == 0 or len(val_span_dataset) == 0:
        print("Error: Empty datasets. Check your data preprocessing.")
        return

    # Create dataloaders
    train_span_dataloader = DataLoader(
        train_span_dataset,
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True
    )
    val_span_dataloader = DataLoader(
        val_span_dataset,
        batch_size=VALID_BATCH_SIZE
    )

    # Create the SpanTagger model
    span_tagger = SpanTagger(SPAN_MODEL_NAME).to(device)

    # Train the model
    print("Training SpanBERT for span tagging...")
    span_tagger = train_span_tagger(
        span_tagger,
        train_span_dataloader,
        val_span_dataloader
    )

    # Save the model
    torch.save(span_tagger.state_dict(), "span_tagger_model.pt")
    print("Saved span tagger model to span_tagger_model.pt")

    # Generate span predictions
    print("\nGenerating span predictions for train data...")
    train_span_preds = generate_span_predictions(span_tagger, train_data, span_tokenizer)

    print("Generating span predictions for validation data...")
    val_span_preds = generate_span_predictions(span_tagger, val_data, span_tokenizer)

    if test_data:
        print("Generating span predictions for test data...")
        test_span_preds = generate_span_predictions(span_tagger, test_data, span_tokenizer)

    # Part 2: Train BERT for spoiler classification
    print("\n=== Part 2: Training BERT for Spoiler Classification ===")

    # Create datasets
    train_class_dataset = SpoilerClassificationDataset(train_data, train_span_preds, bert_tokenizer)
    val_class_dataset = SpoilerClassificationDataset(val_data, val_span_preds, bert_tokenizer)

    print(f"Created classification datasets: {len(train_class_dataset)} training examples, {len(val_class_dataset)} validation examples")

    # Check that the datasets are not empty
    if len(train_class_dataset) == 0 or len(val_class_dataset) == 0:
        print("Error: Empty classification datasets. Check your data preprocessing.")
        return

    # Create dataloaders
    train_class_dataloader = DataLoader(
        train_class_dataset,
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True
    )
    val_class_dataloader = DataLoader(
        val_class_dataset,
        batch_size=VALID_BATCH_SIZE
    )

    # Create the SpoilerClassifier model
    spoiler_classifier = SpoilerClassifier(BERT_MODEL_NAME).to(device)

    # Train the model
    print("Training BERT for spoiler classification...")
    spoiler_classifier = train_spoiler_classifier(
        spoiler_classifier,
        train_class_dataloader,
        val_class_dataloader
    )

    # Save the model
    torch.save(spoiler_classifier.state_dict(), "spoiler_classifier_model.pt")
    print("Saved spoiler classifier model to spoiler_classifier_model.pt")

    # Evaluate on the validation set
    evaluate_classifier(spoiler_classifier, val_class_dataloader)

    # Part 3: Prepare predictions for the test set (if there is one)
    if test_data and 'test_span_preds' in locals():
        # Create the test dataset and predict
        test_class_dataset = SpoilerClassificationDataset(test_data, test_span_preds, bert_tokenizer)
        if len(test_class_dataset) > 0:
            test_class_dataloader = DataLoader(test_class_dataset, batch_size=VALID_BATCH_SIZE)

            # Predict on the test set
            predictions = predict_with_classifier(spoiler_classifier, test_class_dataloader)

            # Convert predictions to labels (0->phrase, 1->passage, 2->multi)
            label_map = {0: 'phrase', 1: 'passage', 2: 'multi'}
            predictions_labels = [label_map[pred] for pred in predictions]

            # Create the submission file (optional)
            valid_test_data = [item for item in test_data if item.get('uuid')]
            if len(valid_test_data) == len(predictions_labels):
                submission_df = pd.DataFrame({
                    'uuid': [item.get('uuid') for item in valid_test_data],
                    'tags': predictions_labels
                })
                submission_df.to_csv('submission.csv', index=False)
                print("Saved predictions to submission.csv")
            else:
                print(f"Warning: Mismatch between test data ({len(valid_test_data)}) and predictions ({len(predictions_labels)})")



## Classification only

Reloads the saved tagger and retrains only stage 2.


In [ ]:
def run_classification_only(train_file, val_file, test_file=None):
    print("Loading data...")
    train_data = load_data(train_file)
    val_data = load_data(val_file)
    test_data = load_data(test_file) if test_file else None

    # Load the already trained SpanTagger model
    print("Loading trained SpanTagger model...")
    try:
        span_tagger = SpanTagger(SPAN_MODEL_NAME).to(device)
        span_tagger.load_state_dict(torch.load("span_tagger_model.pt"))
        print("Successfully loaded span tagger model")
    except Exception as e:
        print(f"Error loading SpanTagger model: {e}")
        return

    # Generate span predictions
    print("\nGenerating span predictions for train data...")
    train_span_preds = generate_span_predictions(span_tagger, train_data, span_tokenizer)

    print("Generating span predictions for validation data...")
    val_span_preds = generate_span_predictions(span_tagger, val_data, span_tokenizer)

    if test_data:
        print("Generating span predictions for test data...")
        test_span_preds = generate_span_predictions(span_tagger, test_data, span_tokenizer)

    # Part 2: Train BERT for spoiler classification
    # Create datasets
    train_class_dataset = SpoilerClassificationDataset(train_data, train_span_preds, bert_tokenizer)
    val_class_dataset = SpoilerClassificationDataset(val_data, val_span_preds, bert_tokenizer)

    print(f"Created classification datasets: {len(train_class_dataset)} training examples, {len(val_class_dataset)} validation examples")

    # Check that the datasets are not empty
    if len(train_class_dataset) == 0 or len(val_class_dataset) == 0:
        print("Error: Empty classification datasets. Check your data preprocessing.")
        return

    # Create dataloaders
    train_class_dataloader = DataLoader(
        train_class_dataset,
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True
    )
    val_class_dataloader = DataLoader(
        val_class_dataset,
        batch_size=VALID_BATCH_SIZE
    )

    # Create the SpoilerClassifier model
    spoiler_classifier = SpoilerClassifier(BERT_MODEL_NAME).to(device)

    # Train the model
    print("Training BERT for spoiler classification...")
    spoiler_classifier = train_spoiler_classifier(
        spoiler_classifier,
        train_class_dataloader,
        val_class_dataloader
    )

    # Save the model
    torch.save(spoiler_classifier.state_dict(), "spoiler_classifier_model.pt")
    print("Saved spoiler classifier model to spoiler_classifier_model.pt")

    # Evaluate on the validation set
    evaluate_classifier(spoiler_classifier, val_class_dataloader)

    # Part 3: Prepare predictions for the test set (if there is one)
    if test_data and 'test_span_preds' in locals():
        # Create the test dataset and predict
        test_class_dataset = SpoilerClassificationDataset(test_data, test_span_preds, bert_tokenizer)
        if len(test_class_dataset) > 0:
            test_class_dataloader = DataLoader(test_class_dataset, batch_size=VALID_BATCH_SIZE)

            # Predict on the test set
            predictions = predict_with_classifier(spoiler_classifier, test_class_dataloader)

            # Convert predictions to labels (0->phrase, 1->passage, 2->multi)
            label_map = {0: 'phrase', 1: 'passage', 2: 'multi'}
            predictions_labels = [label_map[pred] for pred in predictions]

            # Create the submission file (optional)
            valid_test_data = [item for item in test_data if item.get('uuid')]
            if len(valid_test_data) == len(predictions_labels):
                submission_df = pd.DataFrame({
                    'uuid': [item.get('uuid') for item in valid_test_data],
                    'tags': predictions_labels
                })
                submission_df.to_csv('submission.csv', index=False)
                print("Saved predictions to submission.csv")
            else:
                print(f"Warning: Mismatch between test data ({len(valid_test_data)}) and predictions ({len(predictions_labels)})")

## Run the full pipeline


In [ ]:
def main():
    # Set the file paths
    train_file = "train.jsonl"
    val_file = "validation.jsonl"
    test_file = "test.jsonl"

    # Run the full pipeline
    run_full_pipeline(train_file, val_file, test_file)

if __name__ == "__main__":
    main()

Loading data...

=== Part 1: Fine-tuning SpanBERT for Span Tagging ===


Created span tagging datasets: 3199 training examples, 800 validation examples


Training SpanBERT for span tagging...


Epoch 1/3 - Average training loss: 0.2918


Epoch 1/3 - Average validation loss: 0.1981
Validation Token-level Accuracy: 0.9694


Epoch 2/3 - Average training loss: 0.1974


... (12 outputs left out)

Epoch 3/3 - Average training loss: 1.0184


Epoch 3/3 - Average validation loss: 1.0341
Validation Accuracy: 0.4475
Validation Balanced Accuracy: 0.3608
              precision    recall  f1-score   support

      phrase       0.46      0.73      0.56       335
     passage       0.43      0.35      0.39       322
       multi       0.00      0.00      0.00       143

    accuracy                           0.45       800
   macro avg       0.30      0.36      0.32       800
weighted avg       0.36      0.45      0.39       800

Loaded best model with validation loss: 1.0341
Saved spoiler classifier model to spoiler_classifier_model.pt


Evaluation Accuracy: 0.4475
Evaluation Balanced Accuracy: 0.3608
              precision    recall  f1-score   support

      phrase       0.46      0.73      0.56       335
     passage       0.43      0.35      0.39       322
       multi       0.00      0.00      0.00       143

    accuracy                           0.45       800
   macro avg       0.30      0.36      0.32       800
weighted avg       0.36      0.45      0.39       800



Saved predictions to submission.csv


## Second stored run: classification only

Stage 2 retrained with the saved tagger and the settings in the code above. See the README for its result.


In [ ]:
# Run only the classification part
run_classification_only("train.jsonl", "validation.jsonl", "test.jsonl")

Loading data...
Loading trained SpanTagger model...


Successfully loaded span tagger model

Generating span predictions for train data...


Generating span predictions for validation data...


Generating span predictions for test data...



=== Part 2: Training BERT for Spoiler Classification ===


Created classification datasets: 3200 training examples, 800 validation examples


... (9 outputs left out)

Epoch 5/5 - Average training loss: 0.6276


Epoch 5/5 - Average validation loss: 0.8521
Validation Accuracy: 0.6575
Validation Balanced Accuracy: 0.6111
              precision    recall  f1-score   support

      phrase       0.60      0.83      0.70       335
     passage       0.69      0.58      0.63       322
       multi       0.92      0.42      0.58       143

    accuracy                           0.66       800
   macro avg       0.74      0.61      0.64       800
weighted avg       0.70      0.66      0.65       800

Saved spoiler classifier model to spoiler_classifier_model.pt


Evaluation Accuracy: 0.6575
Evaluation Balanced Accuracy: 0.6111
              precision    recall  f1-score   support

      phrase       0.60      0.83      0.70       335
     passage       0.69      0.58      0.63       322
       multi       0.92      0.42      0.58       143

    accuracy                           0.66       800
   macro avg       0.74      0.61      0.64       800
weighted avg       0.70      0.66      0.65       800



Saved predictions to submission.csv
